# Week - 5 project
-	Project day: production-ready pipeline on the House Prices dataset
-	Full Pipeline: custom imputer + scaler + encoder + feature creator + XGBoost
-	Wrap in GridSearchCV, save best pipeline to disk, write a predict.py script


In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Roadmap/Datasets/Housing Price/train.csv")

In [3]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [5]:
null_cols = df.isnull().sum()
null_cols[null_cols > 0].sort_values(ascending=False)

,0
PoolQC,1453
MiscFeature,1406
Alley,1369
Fence,1179
MasVnrType,872
FireplaceQu,690
LotFrontage,259
GarageType,81
GarageYrBlt,81
GarageFinish,81


In [6]:
#dropping cols > 50% null count
df = df.drop(columns=["PoolQC","MiscFeature","Alley", "Fence", "MasVnrType"])

In [7]:
# fill na values of numeric columns with median
for col in df.select_dtypes(include='number').columns:
  df[col] = df[col].fillna(df[col].median())

In [8]:
#Fill na values of categorical columns with mode

for col in df.select_dtypes(include='object').columns:
  df[col] = df[col].fillna(df[col].mode()[0])

In [10]:
df.isnull().sum().sum()

np.int64(0)

In [11]:
numerical_cols = df.select_dtypes(include="number").columns.tolist()

#removing response variable
numerical_cols.remove("SalePrice")

categorical_cols = df.select_dtypes(include="object").columns.tolist()

print(f"Numerical: {len(numerical_cols)} columns")
print(f"Categorical: {len(categorical_cols)} columns")

Numerical: 37 columns
Categorical: 38 columns


In [12]:
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

Pipeline:
- Step 1: FeatureCreator        ← custom transformer (adds 3 features)
- Step 2: ColumnTransformer     ← handles imputing + scaling + encoding
-    ├── numerical_transformer
-    │     ├── SimpleImputer(median)
-    │     └── StandardScaler
-    └── categorical_transformer
-    ├── SimpleImputer(most_frequent)
-    └── OneHotEncoder
  Step 3: XGBoost               ← model

In [14]:
from sklearn.base import BaseEstimator, TransformerMixin


class FeatureCreator(BaseEstimator, TransformerMixin):

  def fit(self, X, y=None):
    return self

  def transform(self, X):
    X= X.copy()
    X['TotalSF'] = X['GrLivArea'] + X['TotalBsmtSF']
    X['OverallQual_sq'] = X['OverallQual'] ** 2
    X['LogLotArea'] = np.log1p(X['LotArea'])
    return X



In [15]:
# add new columns to numeric category

numerical_cols_updated = numerical_cols + ['TotalSF', 'OverallQual_sq', 'LogLotArea']

In [16]:
#Pipeline

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder


pipeline = Pipeline([

    #Step 1: FeatureCreator
    ('feature_creator', FeatureCreator()),

    #Step 2: ColumnTransformer
    ('preprocessor', ColumnTransformer([

        #numerical_transformer
        ('num', Pipeline([
            ('imputer',SimpleImputer(strategy="median")),
            ('scaler', StandardScaler())
        ]), numerical_cols_updated),
        #categorical_transformer
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy="most_frequent")),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_cols)
    ])),

    #Step 3: XGBoost ← model
    ('model', XGBRegressor(random_state = 42))

])

In [18]:
print(pipeline)

Pipeline(steps=[('feature_creator', FeatureCreator()),
                ('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Id', 'MSSubClass',
                                                   'LotFrontage', 'LotArea',
                                                   'OverallQual', 'OverallCond',
                                                   'YearBuilt', 'YearRemodAdd',
                                                   'MasVnrArea', 'BsmtFinSF1',
                                                   'BsmtFinSF2', 'BsmtUnfSF',
                                            

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X,y , test_size=0.3, random_state=42)


pipeline.fit(X_train, y_train)
print("Pipeline fitted successfully")

Pipeline fitted successfully


In [20]:
from sklearn.metrics import mean_squared_error


y_pred = pipeline.predict(X_test)


rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE: ${rmse:,.0f}")

RMSE: $24,614


1. Wrap in GridSearchCV, save best pipeline to disk, write a predict.py script

In [21]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_estimators': [100, 300, 500],
    'model__max_depth': [3, 6],
    'model__learning_rate': [0.01, 0.1]
}

In [25]:
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv = 5,
    scoring= 'neg_mean_squared_error',
    verbose = 1
)

grid_search.fit(X_train, y_train)

print(f"Best Params: {grid_search.best_params_}")
best_rmse = np.sqrt(abs(grid_search.best_score_))
print(f"Best CV RMSE: ${best_rmse:,.0f}")

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best Params: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 300}
Best CV RMSE: $33,653


### best pipeline with joblib

In [26]:
import joblib

joblib.dump(grid_search.best_estimator_, '/content/drive/MyDrive/Colab Notebooks/Roadmap/Week 5 - Feature Engineering + scikit-learn Pipelines/best_pipeline.pkl')
print("Pipeline saved successfully")

Pipeline saved successfully


In [27]:
loaded_pipeline = joblib.load('/content/drive/MyDrive/Colab Notebooks/Roadmap/Week 5 - Feature Engineering + scikit-learn Pipelines/best_pipeline.pkl')

y_pred_loaded = loaded_pipeline.predict(X_test)
rmse_loaded = np.sqrt(mean_squared_error(y_test, y_pred_loaded))
print(f"Loaded pipeline RMSE: ${rmse_loaded:,.0f}")

Loaded pipeline RMSE: $23,976


#### predict.py script

In [30]:
predict_script = '''

import pandas as pd
import numpy as np
import joblib



# Load pipeline
pipeline = joblib.load('best_pipeline.pkl')


# Load new data (raw, unprocessed)
new_data = pd.read_csv('test.csv')


# Store IDs for submission
ids = new_data['Id']


# Predict — pipeline handles all preprocessing automatically
predictions = pipeline.predict(new_data)

# Create submission file
submission = pd.DataFrame({
    'Id': ids,
    'SalePrice': predictions
})

submission.to_csv('predictions.csv', index=False)
print(f"Predictions saved. Sample:")
print(submission.head())

'''

# Save as predict.py
with open('/content/drive/MyDrive/Colab Notebooks/Roadmap/Week 5 - Feature Engineering + scikit-learn Pipelines/predict.py', 'w') as f:
    f.write(predict_script)

print("predict.py saved")

predict.py saved


In [31]:
test_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Roadmap/Datasets/Housing Price/test.csv")

# Run loaded pipeline directly
predictions = loaded_pipeline.predict(test_df)
print(predictions[:5])

[115585.37 163658.61 183867.08 189320.97 180887.92]


In [32]:
#kaggle submission


submission = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': predictions
})
submission.to_csv('/content/drive/MyDrive/Colab Notebooks/Roadmap/Datasets/Housing Price/submission.csv', index=False)
print(submission.head())

     Id      SalePrice
0  1461  115585.367188
1  1462  163658.609375
2  1463  183867.078125
3  1464  189320.968750
4  1465  180887.921875


## Week 5 Project — Production ML Pipeline

### Results
| Step | RMSE |
|------|------|
| Week 4 baseline | 26,305 |
| Full pipeline (no tuning) | 24,614 |
| GridSearchCV best pipeline | 23,976 |

### Architecture
FeatureCreator → ColumnTransformer → XGBoost

### Key decisions
- FeatureCreator: adds TotalSF, OverallQual_sq, LogLotArea
- ColumnTransformer: median imputation + StandardScaler for numeric,
  mode imputation + OneHotEncoder for categorical
- GridSearchCV: tuned learning_rate, max_depth, n_estimators
- joblib: saved full pipeline including preprocessing parameters

### predict.py
Load pipeline → read raw CSV → predict → save submission
One line of predict() handles all preprocessing automatically